# AamirGPT — Fine-tuning Mistral-7B-Instruct with QLoRA
Fine-tunes Mistral-7B to respond to YouTube comments as AamirGPT, a virtual data science consultant.

## Step 0 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q transformers peft datasets accelerate bitsandbytes optimum auto-gptq huggingface_hub
print('All packages installed!')

## Step 1 — Mount Google Drive
Checkpoints and the dataset will be saved to Drive so nothing is lost if the session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/AamirGPT'
CHECKPOINT_DIR    = f'{DRIVE_PROJECT_DIR}/model'
DATASET_DIR       = f'{DRIVE_PROJECT_DIR}/data'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')
print(f'Dataset will be loaded from:  {DATASET_DIR}')

## Step 2 — Keep Colab Alive
This prevents the session from disconnecting when you switch tabs.

In [ ]:
# Paste this in your browser console to prevent idle disconnection:
# function ClickConnect(){
#   console.log('Keeping Colab alive...');
#   document.querySelector('#top-toolbar > colab-connect-button').shadowRoot.querySelector('#connect').click();
# }
# setInterval(ClickConnect, 60000);
#
# OR just keep the Colab tab open and active.
# Training takes ~70 min — Colab disconnects after ~90 min of inactivity.
# Checkpoints are saved to Drive every epoch, so even if it disconnects you won't lose progress.
print('Read the comment above — paste the JS snippet in your browser console to stay connected.')

## Step 3 — Imports

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel
from datasets import load_from_disk
import transformers
import warnings
warnings.filterwarnings('ignore')
from transformers import logging
logging.set_verbosity_error()
print('Imports done.')

## Step 4 — Load Base Model
Downloads ~4GB from HuggingFace. Takes 3–5 minutes on first run.
The model is already 4-bit quantized (GPTQ) — no extra quantization needed.

In [ ]:
model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",        # automatically uses GPU
    trust_remote_code=False,
    revision="main"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
print('Model and tokenizer loaded!')

## Step 5 — Test Base Model (Before Fine-tuning)
See what the model outputs before any training — useful for comparison later.

In [ ]:
model.eval()

comment = "Great content, thank you!"
prompt = f'[INST] {comment} [/INST]'

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=140)

print('--- Base model output (no persona yet) ---')
print(tokenizer.batch_decode(outputs)[0])

## Step 6 — Prepare Model for QLoRA Training

In [ ]:
model.train()
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)  # enables training on quantized weights

# LoRA config — only 0.79% of parameters will be trained
config = LoraConfig(
    r=8,                        # rank of adapter matrices
    lora_alpha=32,              # scaling factor
    target_modules=["q_proj"], # which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

## Step 7 — Load Dataset from Google Drive

In [ ]:
# Load the dataset saved by create-dataset.ipynb
data = load_from_disk(DATASET_DIR)
print(data)

## Step 8 — Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    text = examples["example"]
    tokenizer.truncation_side = "left"  # keep the most recent context
    tokenized_inputs = tokenizer(
        text,
        return_tensors="np",
        truncation=True,
        max_length=512
    )
    return tokenized_inputs

tokenized_data = data.map(tokenize_function, batched=True)

# pad token and data collator
tokenizer.pad_token = tokenizer.eos_token
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)

print('Dataset tokenized!')

## Step 9 — Fine-tune the Model
Training takes ~60–90 minutes on a T4 GPU. Checkpoints are saved to Google Drive after every epoch.

In [ ]:
# Training hyperparameters
lr         = 2e-4
batch_size = 4
num_epochs = 10

training_args = transformers.TrainingArguments(
    output_dir=CHECKPOINT_DIR,              # saves to Google Drive
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",                  # checkpoint saved every epoch
    load_best_model_at_end=True,
    gradient_accumulation_steps=4,          # effective batch size = 4 x 4 = 16
    warmup_steps=2,
    fp16=True,                              # half precision to save VRAM
    optim="paged_adamw_8bit",               # memory-efficient optimizer
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    args=training_args,
    data_collator=data_collator
)

model.config.use_cache = False   # required during training
trainer.train()
model.config.use_cache = True    # re-enable for inference

print('Training complete!')

## Step 10 — Push Fine-tuned Model to HuggingFace Hub
This uploads only the LoRA adapter weights (~few MB), not the full 7B model.

In [ ]:
from huggingface_hub import login

# Get your token from: https://huggingface.co/settings/tokens  (needs 'write' permission)
HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxx"  # <-- paste your HF write token here
HF_USERNAME = "your-hf-username"         # <-- replace with your HuggingFace username

login(HF_TOKEN)
print('Logged in to HuggingFace!')

In [ ]:
model_id = f"{HF_USERNAME}/AamirGPT"

model.push_to_hub(model_id)
trainer.push_to_hub(model_id)

print(f'Model pushed to: https://huggingface.co/{model_id}')

## Step 11 — Test the Fine-tuned Model
Run inference to see how the model responds after fine-tuning.

In [ ]:
instructions_string = """AamirGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. \
It reacts to feedback aptly and ends responses with its signature '–AamirGPT'. \
AamirGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, \
thus keeping the interaction natural and engaging.

Please respond to the following comment.
"""

prompt_template = lambda comment: f'[INST] {instructions_string} \n{comment} \n[/INST]'

def generate_response(comment, max_new_tokens=280):
    prompt = prompt_template(comment)
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"].to("cuda"),
        max_new_tokens=max_new_tokens
    )
    # return only the generated response, not the prompt
    full_output = tokenizer.batch_decode(outputs)[0]
    response = full_output.split('[/INST]')[-1].strip().replace('</s>', '').strip()
    return response

# Test it
print(generate_response("Great content, thank you!"))
print('---')
print(generate_response("What is fat-tailedness?"))

## (Optional) Load Model from HuggingFace Hub
Use this cell in a new session to reload your fine-tuned model without retraining.

In [ ]:
# Uncomment and run this block to load your pushed model from HuggingFace Hub

# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
#
# HF_USERNAME = "your-hf-username"   # <-- your HuggingFace username
# base_model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"
# adapter_model   = f"{HF_USERNAME}/AamirGPT"
#
# model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto", revision="main")
# model = PeftModel.from_pretrained(model, adapter_model)
# tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
# print('Fine-tuned model loaded from Hub!')